In [1]:
from pathlib import Path
import sys
import importlib

# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks
# queremos:  .../traspaso-innominados/functions
PROJECT_ROOT = Path.cwd().resolve().parents[1]
FUNCTIONS_DIR = (PROJECT_ROOT / "functions").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
# Si tu archivo se llama functions.py dentro de FUNCTIONS_DIR, esto funciona.
# Si es paquete "functions/" con __init__.py, esto también funciona.
module_name = "functions"

if module_name in sys.modules:
    fun = importlib.reload(sys.modules[module_name])
else:
    fun = importlib.import_module(module_name)

print("Importado desde:", getattr(fun, "__file__", "<sin __file__>"))

PROJECT_ROOT: C:\Users\aepmvalenzuela\traspaso-innominados
FUNCTIONS_DIR: C:\Users\aepmvalenzuela\traspaso-innominados\functions exists: True
Importado desde: C:\Users\aepmvalenzuela\traspaso-innominados\functions\functions.py


In [2]:
# --- Parámetros ---
server = "SGF1034"
database = "Habitat"
schema = "dbo"

# --- Construir engine + obtener connection_string pyodbc ---
# Usa tus funciones: build_sqlserver_engine (y opcionalmente diagnósticos)

driver = "ODBC Driver 17 for SQL Server"  # o "ODBC Driver 18 for SQL Server"

# 1) ODBC connection string (pyodbc) -> para query_to_df / df_to_db / exec_sql
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# Si tu entorno requiere TLS/cert (muy común con Driver 18), descomenta:
# connection_string += "Encrypt=yes;TrustServerCertificate=yes;"

# 2) SQLAlchemy engine -> usando tu helper (hace test SELECT 1 y crea el engine)
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    # encrypt=True, trust_server_certificate=True,  # si lo necesitas
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise",  # "warn" si prefieres que no explote y solo imprima diagnóstico
)

## CONSOLIDACIÓN: TOTAL_ARAUCANA

In [ ]:
query_monitoreo_1 = """
drop table TOTAL_ARAUCANA_BKP;
"""

fun.exec_sql(query_monitoreo_1, connection_string)

In [ ]:
query_monitoreo_2 = """
SELECT T.*, CONVERT(FLOAT,'0') AS Comision
INTO TOTAL_ARAUCANA_BKP
FROM (
    -- [MARSH]
    SELECT MES_Recaudacion, POLIZA, foliocredito, rutafiliado, MontoBruto, Plazo,
           fechaPrimerVto, FechaUltimoVto, ValorCuota, FechaPrima, Producto,
           FolioOrigen, FechaOrigen, PLAN_TECNICO, PLAZO_CUOTAS, Negocio,
           Prima_Bruta_mensual AS PrimaBruta, Prima_Neta AS PrimaNeta
    FROM MARSH_FINAL_ACUMULADO_BKP

    UNION ALL

    -- [CONOSUR]
    SELECT MES_Recaudacion, POLIZA, foliocredito, rutafiliado, MontoBruto, Plazo,
           fechaPrimerVto, FechaUltimoVto, ValorCuota, FechaPrima, Producto,
           FolioOrigen, FechaOrigen, PLAN_TECNICO, PLAZO_CUOTAS, Negocio,
           PrimaBrutaMensual AS PrimaBruta, PrimaNetaMensual AS PrimaNeta
    FROM CONOSUR_FINAL_ACUMULADO_BKP

    UNION ALL

    -- [VOLVEK_FLUJO2]
    select MES_Recaudacion, POLIZA, foliocredito, rutafiliado, MontoBruto, Plazo,
           fechaPrimerVto, FechaUltimoVto, ValorCuota, NULL AS FechaPrima, Producto,
           FolioOrigen, FechaOrigen, PLAN_TECNICO, PLAZO_CUOTAS, Negocio,
           PrimaBruta, PrimaNeta
    FROM FLUJO2_FINAL_ACUMULADO_BKP

    UNION ALL

    -- [VOLVEK_FLUJO]
    SELECT MES_Recaudacion, POLIZA, foliocredito, rutafiliado, MontoBruto, Plazo,
           fechaPrimerVto, FechaUltimoVto, ValorCuota, NULL AS FechaPrima, Producto,
           FolioOrigen, FechaOrigen, PLAN_TECNICO, PLAZO_CUOTAS, Negocio,
           PrimaBruta, PrimaNeta
    FROM FLUJO_FINAL_ACUMULADO_BKP

    UNION ALL

    -- [VOLVEK_STOCK]
    SELECT MES_Recaudacion, POLIZA, foliocredito, rutafiliado, MontoBruto, Plazo,
           fechaPrimerVto, FechaUltimoVto, ValorCuota, NULL AS FechaPrima, Producto,
           FolioOrigen, FechaOrigen, PLAN_TECNICO, PLAZO_CUOTAS, Negocio,
           PrimaBruta, PrimaNeta
    FROM STOCK_FINAL_ACUMULADO_BKP

    UNION ALL

    -- [VOLVEK_PLANES]
    SELECT MES_Recaudacion, POLIZA, foliocredito, rutafiliado, MontoBruto, Plazo,
           fechaPrimerVto, FechaUltimoVto, ValorCuota, NULL AS FechaPrima, Producto,
           FolioOrigen, FechaOrigen, PLAN_TECNICO, PLAZO_CUOTAS, Negocio,
           PrimaBruta, PrimaNeta
    FROM PLANES_FINAL_ACUMULADO_BKP
) AS T;
"""

fun.exec_sql(query_monitoreo_2, connection_string)

## COMISIONES: UPDATE SOBRE TOTAL_ARAUCANA

In [ ]:
query_monitoreo_3 = """
UPDATE TOTAL_ARAUCANA_BKP
SET Comision = CASE
    WHEN Plan_Tecnico = 4277 THEN 0.0476 * PrimaNeta
    WHEN Plan_Tecnico IN (4331, 4332, 4333, 4334, 4422) THEN 0.0874 * PrimaNeta
    WHEN Plan_Tecnico = 6270 THEN 0.0665 * PrimaNeta
    WHEN Plan_Tecnico IN (6832, 8285) THEN 0.0087 * PrimaNeta
    ELSE Comision -- mantiene
END;

PRINT('fin de la base manual');
"""

fun.exec_sql(query_monitoreo_3, connection_string)

In [ ]:
query_monitoreo_4 = """
-- Validación rápida consolidado
SELECT *
FROM TOTAL_ARAUCANA_BKP;
"""

fun.exec_sql(query_monitoreo_4, connection_string)

## MONITORING: CREACIÓN TABLA DESTINO

In [ ]:
query_monitoreo_5 = """
DROP TABLE Monitoring_LaAraucana_BKP;
"""

fun.exec_sql(query_monitoreo_5, connection_string)

In [ ]:
query_monitoreo_6 = """
CREATE TABLE Monitoring_LaAraucana_BKP
(
   NANOPROC NUMERIC(4,0),
   NMESPROC NUMERIC(2,0),
   CCODLNRO VARCHAR(11),
   LN NUMERIC(9,0),
   NNUMDOCU NUMERIC(20,0),
   NNUMENDO BIGINT,
   NNUMITEM BIGINT,
   NNUMCERT BIGINT,
   PROPUESTA BIGINT,
   CCODCOBE VARCHAR(5),
   CCODRAMO VARCHAR(2),
   NCORASVS NUMERIC(5,0),
   PLAN_TECNICO NUMERIC(5,0),
   RAMO_IBNR VARCHAR(50),
   PARTNER_SUCURSAL VARCHAR(100),
   BU VARCHAR(70),
   COBERTURA1 VARCHAR(50),
   COBERTURA2 VARCHAR(80),
   TIPO_COBERTURA CHAR(40),
   CONTRATO_REAS CHAR(50),
   MCA_VIG VARCHAR(2),
   TIPO VARCHAR(6),
   RUT_INT NUMERIC(10,0),
   PARTNER_PYG NVARCHAR(255),
   PLAN_COMERCIAL_GOL NUMERIC(5,0),
   CANAL_GOL NVARCHAR(35),
   SUCURSAL_GOL VARCHAR(60),
   RUT_INTERMEDIARIO_GOL NUMERIC(10,0),
   RUT_EJECUTIVO_GOL NUMERIC(10,0),
   PARTNER_GOL VARCHAR(100),
   BU_PYG NVARCHAR(50),
   ZONA_SUCURSAL NVARCHAR(50),
   RAMO_IBNR_ORIG NVARCHAR(50),
   COBERTURA_AGRUP NVARCHAR(50),
   NFECEMIS DATE,
   NFEINVIG DATE,
   NFETEVIG DATE,
   CCOMOORI VARCHAR(3),
   TC INT,
   TC_UF FLOAT,
   PRIMA FLOAT,
   COMISION FLOAT,
   PRIMA_CLP FLOAT,
   COMISION_CLP FLOAT,
   PRIMA_UF FLOAT,
   COMISION_UF FLOAT
);
"""

fun.exec_sql(query_monitoreo_6, connection_string)

## MONITORING: INSERT DETALLE (por cobertura real desde EMISION_DEVENGADA_PPI)

In [ ]:
query_monitoreo_7 = """
INSERT INTO Monitoring_LaAraucana_BKP
SELECT
    YEAR(f.PER_ANT) AS NANOPROC,
    MONTH(f.PER_ANT) AS NMESPROC,
    a.CCODLNRO,
    a.LN,
    b.POLIZA AS NNUMDOCU,
    0 AS NNUMENDO,
    CONVERT(BIGINT, b.foliocredito) AS NNUMITEM,
    b.rutafiliado AS NNUMCERT,
    CONVERT(BIGINT, b.foliocredito) AS PROPUESTA,
    a.CCODCOBE,
    a.CCODRAMO,
    a.NCORASVS,
    a.PLAN_TECNICO,
    a.RAMO_IBNR,
    a.PARTNER_SUCURSAL,
    a.BU,
    a.COBERTURA1,
    a.COBERTURA2,
    a.TIPO_COBERTURA,
    a.CONTRATO_REAS,
    a.MCA_VIG,
    a.TIPO,
    a.RUT_INT,
    a.PARTNER_PYG,
    a.PLAN_COMERCIAL_GOL,
    a.CANAL_GOL,
    a.SUCURSAL_GOL,
    a.RUT_INTERMEDIARIO_GOL,
    a.RUT_EJECUTIVO_GOL,
    a.PARTNER_GOL,
    a.BU_PYG,
    a.ZONA_SUCURSAL,
    a.RAMO_IBNR_ORIG,
    a.COBERTURA_AGRUP,
    f.PER AS NFECEMIS,
    f.PER_ANT AS NFEINVIG,
    DATEADD(day, 1, f.PER) AS NFETEVIG,
    'CLP' AS CCOMOORI,
    0 AS TC,
    c.TC_UF,
    b.PrimaNeta AS PRIMA,
    b.Comision * -1 AS COMISION,
    b.PrimaNeta AS PRIMA_CLP,
    b.Comision * -1 AS COMISION_CLP,
    (b.PrimaNeta / c.TC_UF) AS PRIMA_UF,
    (b.Comision * -1 / c.TC_UF) AS COMISION_UF
FROM TOTAL_ARAUCANA_BKP b
LEFT JOIN (
    SELECT
        NNUMDOCU,
        CCODLNRO,
        LN,
        CCODCOBE,
        CCODRAMO,
        NCORASVS,
        PLAN_TECNICO,
        RAMO_IBNR,
        PARTNER_SUCURSAL,
        BU,
        COBERTURA1,
        COBERTURA2,
        TIPO_COBERTURA,
        CONTRATO_REAS,
        MCA_VIG,
        CCOMOORI,
        TIPO,
        RUT_INT,
        PARTNER_PYG,
        PLAN_COMERCIAL_GOL,
        CANAL_GOL,
        SUCURSAL_GOL,
        RUT_INTERMEDIARIO_GOL,
        RUT_EJECUTIVO_GOL,
        PARTNER_GOL,
        BU_PYG,
        ZONA_SUCURSAL,
        RAMO_IBNR_ORIG,
        COBERTURA_AGRUP,
        count(NNUMDOCU) AS casos
    FROM [Habitat].[dbo].[EMISION_DEVENGADA_PPI]
    WHERE nnumdocu IN (4780715, 4780716, 4780717, 4780718, 4780719, 4659577, 4668266, 5698774, 6354562, 7561166)
    GROUP BY
        NNUMDOCU, CCODLNRO, LN, CCODCOBE, CCODRAMO, NCORASVS, PLAN_TECNICO, RAMO_IBNR, PARTNER_SUCURSAL,
        BU, COBERTURA1, COBERTURA2, TIPO_COBERTURA, CONTRATO_REAS, MCA_VIG, CCOMOORI, TIPO, RUT_INT,
        PARTNER_PYG, PLAN_COMERCIAL_GOL, CANAL_GOL, SUCURSAL_GOL, RUT_INTERMEDIARIO_GOL,
        RUT_EJECUTIVO_GOL, PARTNER_GOL, BU_PYG, ZONA_SUCURSAL, RAMO_IBNR_ORIG, COBERTURA_AGRUP
) a
    ON b.POLIZA = a.NNUMDOCU
LEFT JOIN (SELECT * FROM Sura_Periodo) f
    ON b.MES_Recaudacion = f.MES_REF
LEFT JOIN (SELECT *, CAST(fecha as DATE) AS fecha_date FROM TC_DIARIO) c
    ON f.PER = c.fecha_date;
"""

fun.exec_sql(query_monitoreo_7, connection_string)

## (OPCIONAL) CONSULTA SOPORTE UF

In [ ]:
query_monitoreo_8 = """
SELECT *
FROM TC_DIARIO;
"""

fun.exec_sql(query_monitoreo_8, connection_string)

## MONITORING: INSERT TOTAL (agrega fila "TOTAL" por póliza)

In [ ]:
query_monitoreo_9 = """
INSERT Monitoring_LaAraucana_BKP
SELECT
    YEAR(f.PER_ANT) as NANOPROC,
    MONTH(f.PER_ANT) as NMESPROC,
    a.CCODLNRO,
    a.LN,
    b.POLIZA as NNUMDOCU,
    0 as NNUMENDO,
    CONVERT(BIGINT, b.foliocredito) as NNUMITEM,
    b.rutafiliado as NNUMCERT,
    CONVERT(BIGINT, b.foliocredito) as PROPUESTA,
    1 as CCODCOBE,
    a.CCODRAMO,
    99 as NCORASVS,
    a.PLAN_TECNICO,
    a.RAMO_IBNR,
    a.PARTNER_SUCURSAL,
    a.BU,
    'TOTAL' as COBERTURA1,
    'TOTAL' as COBERTURA2,
    a.TIPO_COBERTURA,
    a.CONTRATO_REAS,
    a.MCA_VIG,
    a.TIPO,
    a.RUT_INT,
    a.PARTNER_PYG,
    a.PLAN_COMERCIAL_GOL,
    a.CANAL_GOL,
    a.SUCURSAL_GOL,
    a.RUT_INTERMEDIARIO_GOL,
    a.RUT_EJECUTIVO_GOL,
    a.PARTNER_GOL,
    a.BU_PYG,
    a.ZONA_SUCURSAL,
    a.RAMO_IBNR_ORIG,
    'TOTAL' as COBERTURA_AGRUP,
    f.PER as NFECEMIS,
    f.PER_ANT as NFEINVIG,
    DATEADD(day, 1, f.PER) AS NFETEVIG,
    'CLP' as CCOMOORI,
    0 as TC,
    c.TC_UF,
    b.PrimaNeta as PRIMA,
    b.Comision as COMISION,
    b.PrimaNeta as PRIMA_CLP,
    b.Comision * -1 as COMISION_CLP,
    (b.PrimaNeta / c.TC_UF) as PRIMA_UF,
    (b.Comision * -1 / c.TC_UF) as COMISION_UF
FROM TOTAL_ARAUCANA b
LEFT JOIN (
    SELECT
        NNUMDOCU,
        CCODLNRO,
        LN,
        CCODRAMO,
        PLAN_TECNICO,
        RAMO_IBNR,
        PARTNER_SUCURSAL,
        BU,
        TIPO_COBERTURA,
        CONTRATO_REAS,
        MCA_VIG,
        CCOMOORI,
        TIPO,
        RUT_INT,
        PARTNER_PYG,
        PLAN_COMERCIAL_GOL,
        CANAL_GOL,
        SUCURSAL_GOL,
        RUT_INTERMEDIARIO_GOL,
        RUT_EJECUTIVO_GOL,
        PARTNER_GOL,
        BU_PYG,
        ZONA_SUCURSAL,
        RAMO_IBNR_ORIG,
        count(NNUMDOCU) as casos
    FROM [Habitat].[dbo].[EMISION_DEVENGADA_PPI]
    WHERE nnumdocu IN (4780715, 4780716, 4780717, 4780718, 4780719, 4659577, 4668266, 5698774, 6354562, 7561166)
    GROUP BY
        NNUMDOCU, CCODLNRO, LN, CCODRAMO, NCORASVS, PLAN_TECNICO, RAMO_IBNR, PARTNER_SUCURSAL,
        BU, TIPO_COBERTURA, CONTRATO_REAS, MCA_VIG, CCOMOORI, TIPO, RUT_INT,
        PARTNER_PYG, PLAN_COMERCIAL_GOL, CANAL_GOL, SUCURSAL_GOL, RUT_INTERMEDIARIO_GOL,
        RUT_EJECUTIVO_GOL, PARTNER_GOL, BU_PYG, ZONA_SUCURSAL, RAMO_IBNR_ORIG
) a
    ON b.POLIZA = a.NNUMDOCU
LEFT JOIN (select * from Sura_Periodo) f
    ON b.MES_Recaudacion = f.MES_REF
LEFT JOIN (select *, CAST(fecha as DATE) as fecha_date from TC_DIARIO) c
    ON f.PER = c.fecha_date;
"""

fun.exec_sql(query_monitoreo_9, connection_string)

## MONITORING: UPDATE FINAL PARTNER

In [ ]:
query_monitoreo_10 = """
UPDATE Monitoring_LaAraucana_BKP
SET PARTNER_SUCURSAL = 'La Araucana',
    PARTNER_GOL      = 'La Araucana';
"""

fun.exec_sql(query_monitoreo_10, connection_string)

# Agregado de suma asegurada desde Cesantía

In [ ]:
query_monitoreo_11 = """
DROP TABLE IF EXISTS #Oficio_LA_Araucana_PPI_BKP;
"""

fun.exec_sql(query_monitoreo_11, connection_string)

In [ ]:
query_monitoreo_12 = """
/*--------------------------------------------------------------
 1. CREACIÓN DE TABLA TEMPORAL BASE
    - Extrae información desde Monitoring_LaAraucana
    - Excluye registros agregados (COBERTURA_AGRUP = 'TOTAL')
--------------------------------------------------------------*/

SELECT 
    -- Mes de referencia en formato AAAAMM
    (NANOPROC * 100 + NMESPROC) AS mes_referencia, 

    -- Tipo de documento asegurado (valor fijo)
    1 AS COD_TIPO_DOCUMENTO_ASEGURADO, 

    -- RUT del asegurado
    NNUMCERT AS RUT, 

    -- Número de documento / póliza
    NNUMDOCU, 

    -- Ítem de la póliza
    NNUMITEM, 

    -- Plan técnico del producto
    PLAN_TECNICO,

    -- Código de ramo
    NCORASVS AS COD_RAMO, 

    -- Prima expresada en UF
    PRIMA_UF, 

    -- Periodicidad de pago (valor fijo)
    3 AS COD_PERIODICIDAD_PAGO, 

    -- Tipo de pago (valor fijo)
    9 AS COD_TIPO_PAGO
INTO #Oficio_LA_Araucana_PPI_BKP
FROM Monitoring_LaAraucana_BKP
WHERE COBERTURA_AGRUP != 'TOTAL';
"""

fun.exec_sql(query_monitoreo_12, connection_string)

In [ ]:
query_monitoreo_13 = """
/*--------------------------------------------------------------
 2. DEPURACIÓN DE REGISTROS POR MES DE CORTE
    - Regla especial para nnumdocu específicos
    - Regla general para el resto de documentos
--------------------------------------------------------------*/

DELETE FROM #Oficio_LA_Araucana_PPI_BKP
WHERE 
      nnumdocu IN (6354562 , 7561166) 
      AND mes_referencia < 202411
   OR nnumdocu NOT IN (6354562 , 7561166) 
      AND mes_referencia < 202412;
"""

fun.exec_sql(query_monitoreo_13, connection_string)

In [ ]:
query_monitoreo_14 = """
DROP TABLE IF EXISTS #Oficio_LA_Araucana_PPI2_BKP;
"""

fun.exec_sql(query_monitoreo_14, connection_string)

In [ ]:
query_monitoreo_15 = """
/*--------------------------------------------------------------
 3. CONSOLIDACIÓN DE INFORMACIÓN
    - Se agrupa por póliza / ítem / plan
    - Se calcula:
        * Último mes informado (MAX mes_referencia)
        * Prima directa acumulada (SUM PRIMA_UF)
--------------------------------------------------------------*/

SELECT 
    -- Último mes informado por póliza/ítem
    MAX(mes_referencia) AS max_mes_referencia, 

    COD_TIPO_DOCUMENTO_ASEGURADO, 
    RUT, 
    NNUMDOCU, 
    NNUMITEM, 
    PLAN_TECNICO,
    COD_RAMO, 

    -- Prima directa acumulada
    SUM(PRIMA_UF) AS PRIMA_DIRECTA, 

    COD_PERIODICIDAD_PAGO, 
    COD_TIPO_PAGO
INTO #Oficio_LA_Araucana_PPI2_BKP
FROM #Oficio_LA_Araucana_PPI_BKP
GROUP BY     
    COD_TIPO_DOCUMENTO_ASEGURADO, 
    RUT, 
    NNUMDOCU, 
    NNUMITEM, 
    PLAN_TECNICO,
    COD_RAMO, 
    COD_PERIODICIDAD_PAGO, 
    COD_TIPO_PAGO;
"""

fun.exec_sql(query_monitoreo_15, connection_string)

In [ ]:
query_monitoreo_16 = """
DROP TABLE IF EXISTS Oficio_LA_Araucana_PPI202512_BKP;
"""

fun.exec_sql(query_monitoreo_16, connection_string)

In [ ]:
query_monitoreo_17 = """
/*--------------------------------------------------------------
 4. CONSTRUCCIÓN DE TABLA FINAL PARA OFICIO CMF
    - Se agregan datos de la compañía
    - Se determina vigencia según mes de corte
    - Se calcula monto asegurado directo en CLP
--------------------------------------------------------------*/

SELECT 
    -- Identificación de la compañía aseguradora
    99017000 AS RUT_COMPANIA, 
    2 AS DV_RUT_COMPANIA,
    'Seguros Suramericana S.A.' AS NOMBRE_COMPANIA,

    -- Datos consolidados
    a.*, 

    -- RUT del contratante (valor fijo)
    70016160 AS RUT_CONTRATANTE,    
    9 AS DV_CONTRANTE,

    /*----------------------------------------------------------
      Determinación de vigencia:
      - Documentos especiales: vigentes si último mes = 202510
      - Resto: vigentes si último mes = 202511
    ----------------------------------------------------------*/
    CASE 
        WHEN nnumdocu IN (6354562 , 7561166) 
             AND max_mes_referencia = 202510 THEN 1 
        WHEN nnumdocu NOT IN (6354562 , 7561166) 
             AND max_mes_referencia = 202511 THEN 1 
        ELSE 0  
    END AS vigentes,

    /*----------------------------------------------------------
      Cálculo de monto asegurado directo en CLP
      - Depende del plan técnico
      - Se multiplica ValorCuota por un factor fijo
    ----------------------------------------------------------*/
    CASE 
        WHEN a.PLAN_TECNICO = 4331 THEN b.ValorCuota * 3 
        WHEN a.PLAN_TECNICO = 4332 THEN b.ValorCuota * 6
        WHEN a.PLAN_TECNICO = 4333 THEN b.ValorCuota * 8
        WHEN a.PLAN_TECNICO = 4334 THEN b.ValorCuota * 12
        ELSE b.ValorCuota * 4 
    END AS MONTO_ASEGURADO_DIRECTO_CLP,

    -- Subdivisión informada a CMF
    '3B' AS Subdiv, 

    -- Código de ramo CMF
    33 AS RAMO_CMF
INTO Oficio_LA_Araucana_PPI202512_BKP
FROM #Oficio_LA_Araucana_PPI2_BKP a
LEFT JOIN TOTAL_ARAUCANA_BKP b
    -- Se cruza por mes de recaudación
    ON a.max_mes_referencia = b.MES_Recaudacion
   -- Ítem / folio de crédito
   AND a.NNUMITEM = b.foliocredito
   -- Número de póliza
   AND a.nnumdocu = b.POLIZA;
"""

fun.exec_sql(query_monitoreo_17, connection_string)

In [ ]:
query_monitoreo_18 = """
/*--------------------------------------------------------------
 5. CONSULTA A BASE EXTERNA (LINKED SERVER)
    - Permite validar información de documentos específicos
    - Uso informativo / control
--------------------------------------------------------------*/

SELECT *
FROM OPENQUERY (
    SOL, 
    'SELECT * 
     FROM wrkroyal.ofi113857 
     WHERE DOCUMENTO IN (
        4668266, 4780716, 4780717, 4780718, 4780719,
        5698774, 6354562, 7561166, 4659577, 4780715
     )'
);
"""

fun.exec_sql(query_monitoreo_18, connection_string)

In [ ]:
query_monitoreo_19 = """
/*--------------------------------------------------------------
 6. CONTROL FINAL – RESUMEN DE REGISTROS
    - Cuenta pólizas vigentes vs no vigentes
    - Validación previa al envío CMF
--------------------------------------------------------------*/

SELECT 
    VIGENTES,
    COUNT(NNUMDOCU) AS REGISTROS
FROM Oficio_LA_Araucana_PPI202512_BKP
GROUP BY VIGENTES;
"""

fun.exec_sql(query_monitoreo_19, connection_string)